# Lab 02: Data Acquisition and Cleaning
**Individual · 20 points**

## Your first assignment: campus bike analyst
You have joined the campus bike program. Your supervisor wants to compare trip durations and distances across departure stations, but the trip log needs checking first.

Prepare the data for that future comparison: inspect the log, diagnose problems, clean a copy, try the two scaling methods from lecture, and save a reusable table. You do not need to compare stations or build a model.

**How to work:** follow **Learn → Code → Check → Explain**. Run the provided examples, then write your own code in the exercise cells. Read the output before moving on. Write Responses A, B, and C in the three marked answer cells together under **Written responses** near the end. Use Colab’s Table of contents to jump there.

### Learning objectives
- **1B2a:** Treat missing, duplicate, and inconsistent values while preserving raw data.
- **1B2b:** Check the cleaned result and explain a consequence of cleaning.
- **1B2c:** Export a CSV and reload it to check values and counts.
- Compare min-max normalization with z-score standardization, as in Week 1 lecture.

The [lab webpage](https://rhit-csse.github.io/cssema384/labs/lab02.html) provides an overview and troubleshooting.

## Independent Coding Practice: AI Assistance Off

Write your own code. Do not use ChatGPT, Gemini, Copilot, or AI-generated autocomplete for these exercises.

In Colab, open **Edit > Settings > AI Assistance**. Where available, turn off **Show AI-powered inline completions** and turn on **Hide generative AI features**. Labels may vary; ask for help if needed. Avoid AI chat and code-generation buttons.

Use readings, documentation, provided examples, and ordinary editor features such as indentation and non-AI completion. Ask the instructor when stuck.

## Set up your Colab copy
1. Choose **File → Save a copy in Drive**, rename it, and work in that copy.
2. Click **Connect**. Run code with its play button or **Shift+Enter**; wait for completion and read the output underneath.
3. Run the loading cell below. It reads the course's hosted CSV; no upload is needed for this lab.

**Colab reminders:** write Python below comments in code cells. A line starting with `#` is a note, not an action. Use `display(...)` for each result you want to see; only the last bare expression displays automatically. Double-click a Response text cell to write, then press **Shift+Enter** to display it.

Run top to bottom. After changing an earlier cell, rerun it and the cells that depend on it. A runtime restart clears variables, even if old outputs remain visible. Save regularly with **File → Save** or Ctrl/Cmd+S.

**Completing skeletons:** replace every `____` with your own Python code. A blank may stand for a method name, column name, value, or expression; use the nearby example. Quoted role names are strings. Incomplete skeletons will raise errors until you fill them in. Leave the provided variable names unchanged.

In [ ]:
import pandas as pd

dataset_url = "https://raw.githubusercontent.com/RHIT-CSSE/cssema384/main/labs/week01/week01-campus-bike-trips.csv"
raw = pd.read_csv(dataset_url)
print("Trip log loaded.")

## What does a trip record mean?
These are fictional trips with intentional quality problems. One row represents one trip. Blank fields mean unknown.

| Column | Meaning | Valid values |
|---|---|---|
| `trip_id` | Unique trip identifier | One ID per trip |
| `station` | Departure station | `library`, `science`, `residence` |
| `rider_type` | Rider category | `student`, `staff` |
| `duration_min` | Duration in minutes | Greater than 0 |
| `distance_km` | Distance in kilometers | Greater than 0 |

Capitalization and outer spaces do not change a category's meaning. Both measurements are required. Missing rider types are allowed. Keep the original `raw` table unchanged; do not fill missing values with guesses (**imputation**).

## 1. What is in the file? · 3 points
**Learn:** `shape` gives `(rows, columns)`, `head()` previews records, and `dtypes` shows stored types. Run this separate parcel example; later examples reuse it.

In [ ]:
parcels = pd.DataFrame({
    "parcel_id": [501, 502, 502, 503],
    "service": [" Standard ", "EXPRESS", "EXPRESS", None],
    "mass_g": [100, -5, -5, None]
})
display(parcels.shape)
display(parcels.head())
display(parcels.dtypes)

**Learn:** A column's role describes its meaning:
- `identifier`: a record code; averaging codes is not meaningful.
- `categorical`: a label that groups records, even when stored as a numeric code.
- `measurement`: a quantity with units; differences and averages have meaning.

For the example, `parcel_roles = {"parcel_id": "identifier", "service": "categorical", "mass_g": "measurement"}`. A dictionary uses `{key: value}` pairs separated by commas. Numeric parcel IDs are identifiers, even though pandas stores them as numbers.

**Code:** How large is the bike table? What do its first records look like? Display `raw.shape`, `raw.head()`, and `raw.dtypes`. Create `feature_roles`, using each of the five exact column names as a key and one of the three role strings as its value. Roles may repeat.

**Check:** Your dictionary has five keys matching the bike columns. Use the data dictionary to decide roles; do not rely on stored types alone. You will revisit these choices before scaling.

In [ ]:
# Replace each ____ with a method name, attribute, or expression.
display(raw.____)       # (rows, columns)
display(raw.____())     # First five rows
display(raw.____)       # Stored data types

feature_roles = {
    "trip_id": ____,
    "station": ____,
    "rider_type": ____,
    "duration_min": ____,
    "distance_km": ____
}
display(feature_roles)

## 2. Can we use the trip log as it is? · 4 points
Your supervisor needs to know which problems could distort the comparison. Answer the four questions below using unchanged `raw`. Diagnose first; clean in Section 3.

### 2a. Which information is missing?
**Learn:** `parcels["service"].isna().sum()` returns `1`. `isna()` marks missing entries; `sum()` counts them. `parcels.isna().sum()` applies the same operation to every column.

The second result is a **Series**, a labeled sequence of values. It looks like:
```text
parcel_id    0
service      1
mass_g       1
```
**Code:** Count missing entries in every bike column, save the Series as `missing_counts`, and display it.

**Check:** Each of the five column names appears beside its own count. A missing measurement is unknown, not zero.

In [ ]:
# Count missing entries in every column; keep the result as a Series.
missing_counts = raw.____().sum()
display(missing_counts)

### 2b. Were records copied twice?
**Learn:** `parcels.duplicated().sum()` returns `1`: one extra copy of a complete row. If a row appears three times, there are two extra copies.

To inspect all records sharing a parcel ID:
```python
repeated = parcels["parcel_id"].duplicated(keep=False)
display(parcels[repeated])
```
`keep=False` marks every member of a repeated-ID group. Matching IDs alone do not prove that all other values match.

**Code:** Define and display `duplicate_count`, the number of extra exact duplicate rows in `raw`. Display the complete bike records with repeated `trip_id` values.

**Check:** Compare every column in those records. Do they describe the same trip, or disagree? Remove a duplicate only after this comparison.

For this dataset, inspect the repeated-ID records before removing any row. If records with the same ID disagree, removing one arbitrarily is not a valid cleaning rule.

In [ ]:
# Count extra exact copies, then inspect every member of repeated-ID groups.
duplicate_count = raw.____().sum()
repeated_ids = raw["trip_id"].duplicated(keep=____)
display(duplicate_count)
display(raw[____])

### 2c. Which measurements are impossible?
**Learn:** `(parcels["mass_g"] <= 0).sum()` returns `2`. It counts recorded masses at or below zero, separately from missing values. To select several columns, use a list inside the brackets: `table[["mass_g", "width_cm"]]`.

**Code:** Count values at or below zero in `duration_min` and `distance_km`. Save the two labeled counts as a Series named `invalid_counts` and display it.

**Check:** You have one count for each measurement. Missing entries belong in 2a, not in these invalid-value counts.

In [ ]:
# Select both required measurements and write the nonpositive-value condition.
measurement_columns = [____, ____]
invalid_counts = (____).sum()
display(invalid_counts)

### 2d. Are group labels consistent?
**Learn:** `parcels["service"].dropna().unique().tolist()` gives `[" Standard ", "EXPRESS"]`. The methods exclude missing entries, find distinct values, then convert them to a list. Spaces and capitalization remain visible.

A dictionary can hold named lists:
```python
parcel_labels = {
    "service": parcels["service"].dropna().unique().tolist()
}
```
**Code:** Create `category_values` with keys `station` and `rider_type`. For each, store the distinct known labels exactly as recorded. Display the dictionary without changing `raw`.

**Check:** Can you spot labels that should describe the same group? Preserve those differences until you begin cleaning.

In [ ]:
# Keep original labels, excluding missing entries.
category_values = {
    "station": raw["station"].____().unique().____(),
    "rider_type": raw["rider_type"].____().unique().____()
}
display(category_values)

## 3. How should we handle the problems? · 5 points
Apply the bike program's policy in the six steps below. Keep all five original columns and their units. Do not impute missing values.

### 3a. How will you preserve the original?
**Learn:** `tidy_parcels = parcels.copy()` creates a separate working table.

**Code:** Create `clean` from a copy of `raw`.

**Check:** Both tables initially have the same dimensions. Later edits should affect only `clean`.

In [ ]:
# Create a separate working table; raw must remain unchanged.
clean = raw.____()
display(clean.shape)

### 3b. Which labels should be combined?
**Learn:** For a parcel copy, `tidy_parcels["service"] = tidy_parcels["service"].str.strip().str.lower()` changes `" Standard "` to `"standard"`. Assign the result back to keep the change. Missing labels remain missing.

**Code:** Remove outer spaces and lowercase `station` and `rider_type` in `clean`.

**Check:** Display their known labels again. Equivalent spellings should now match. Do not turn missing labels into the text `"nan"` or `"None"`.

In [ ]:
# Complete both transformations and assign the results back.
clean["station"] = clean["station"].str.____().str.____()
clean["rider_type"] = clean["rider_type"].str.____().str.____()
display(clean["station"].dropna().unique())
display(clean["rider_type"].dropna().unique())

### 3c. Which repeated record should be removed?
**Learn:** After confirming identical records, `tidy_parcels = tidy_parcels.drop_duplicates()` keeps one copy of each complete row.

**Code:** Remove the confirmed extra exact duplicate from `clean`. Do not combine different trips just because their measurements match.

**Check:** No extra exact copies should remain. Check repeated trip IDs again. After removing the confirmed exact duplicate in this file, the retained IDs should also be unique; do not drop records just to force this check to pass.

In [ ]:
# Remove only extra copies of confirmed identical records.
clean = clean.____()
print("Extra exact copies:", clean.duplicated().sum())
print("Repeated IDs remain:", clean["trip_id"].duplicated().any())

### 3d. How should impossible measurements be represented?
**Learn:** `tidy_parcels["mass_g"] = tidy_parcels["mass_g"].mask(tidy_parcels["mass_g"] <= 0)` replaces nonpositive mass with missing values. `mask(condition)` changes values where the condition is true; it does not remove rows.

**Code:** Mark nonpositive `duration_min` and `distance_km` values missing in `clean`.

**Check:** Nonpositive counts should now be zero. Missing counts may have increased. You have not guessed any replacement measurements.

In [ ]:
# Complete the condition for each required measurement.
clean["duration_min"] = clean["duration_min"].mask(____)
clean["distance_km"] = clean["distance_km"].mask(____)
display(clean[["duration_min", "distance_km"]].isna().sum())

### 3e. Which trips can still be used?
**Learn:** `tidy_parcels.dropna(subset=["mass_g"])` keeps rows with known mass. `subset` names the required columns. With two columns, the default rule removes a row missing **either** one.

**Code:** Remove rows missing duration or distance. Missing rider type alone is not a reason to remove a trip; leave that label missing.

**Check:** Both required measurements are now present and positive. Avoid bare `dropna()`, which would also remove trips for missing optional labels.

In [ ]:
# Name the required columns; do not remove a trip for rider_type alone.
clean = clean.dropna(subset=[____, ____]).copy()
display(clean.shape)

### 3f. What changed?
**Code:** Display the first rows of `clean` and the row counts of both `raw` and `clean`. Revisit the original problem rows in `raw`; their values should be unchanged.

**Check:** You still have five columns with the original units. Each removed row should be explained by the duplicate or required-measurement rule.

**Explain:** Complete Response A in the **Written responses** section after Section 5.

In [ ]:
print("Original rows:", ____)
print("Cleaned rows:", ____)
display(clean.____())
# Inspect the original problem rows again to check that raw is unchanged.
display(____)

## 4. How can we express measurements on comparable scales? · 4 points
Duration is measured in minutes and distance in kilometers. As in lecture, try two transformations after cleaning; keep original measurements so their units remain available.

**Learn:** Min-max **normalization** expresses position within the observed range: `(x - x.min()) / (x.max() - x.min())`. Z-score **standardization** expresses position relative to the mean and spread: `(x - x.mean()) / x.std()`.

For a separate example with masses 100, 200, and 600 grams:

In [ ]:
masses = pd.Series([100, 200, 600])
mass_range = (masses - masses.min()) / (masses.max() - masses.min())
mass_z = (masses - masses.mean()) / masses.std()
display(mass_range)
display(mass_z)

**Read the example:** Min-max values are 0, 0.2, and 1. A value of 0.5 would be 350 grams, halfway through the range. A z-score of 0 would be 300 grams, the mean. These are different reference points.

Use ordinary pandas `.std()` for standard deviation, both in the formula and when checking the result. The familiar z-score formula is `(value - mean) / standard deviation`.

**Code:** Revisit `feature_roles`: transform measurements, not trip IDs or category labels. Starting from the provided copy below, add `duration_min_scaled` for min-max duration, `distance_km_scaled` for min-max distance, and `duration_min_z` for duration z-scores. Calculate each from `clean`, not an already transformed column. The cleaned measurements are nonconstant.

In [ ]:
scaled = clean.copy()

In [ ]:
duration = clean["duration_min"]
# Min-max: (value - minimum) / (maximum - minimum)
scaled["duration_min_scaled"] = (duration - ____) / (____ - ____)

In [ ]:
distance = clean["distance_km"]
# Apply the same min-max formula to distance.
scaled["distance_km_scaled"] = (distance - ____) / (____ - ____)

In [ ]:
# Z-score: (value - mean) / standard deviation
scaled["duration_min_z"] = (duration - ____) / ____
# Display the min/max ranges, z-score mean and standard deviation, and first rows.
display(____)

**Check:** Display `scaled.head()`, the minimum and maximum of both min-max columns, and the z-score column's mean and `.std()`. Expect ranges from 0 to 1, a mean near 0, and standard deviation near 1. Tiny rounding differences are normal.

**Explain:** Complete Response B in the **Written responses** section after Section 5.

## 5. Is the dataset ready to hand over? · 4 points
Your supervisor should receive a checked table that can be reopened.

**Learn:** A Boolean check returns `True` or `False`. For example:
```python
parcels["parcel_id"].is_unique        # False: an ID repeats
(parcels["mass_g"] > 0).all()        # False: not every mass is positive
parcels["service"].dropna().isin(["standard", "express"]).all()
```
`is_unique` checks repeats, `all()` asks whether every result is true, and `isin(...)` checks membership in an allowed list. The last example is false before labels are cleaned. Use `notna().all()` to check presence. For multiple selected columns, `.all().all()` combines both rows and columns into one Boolean.

**Code:** Create and display `validation_checks`, with keys:
- `unique_ids`: all trip IDs in `clean` are unique.
- `positive_measurements`: duration and distance are present and positive for every retained trip.
- `allowed_categories`: all stations and all known rider labels match the data dictionary. Ignore missing rider labels for this category check.

**Check:** Each result should be `True` and agree with the table. If a result is false, revisit cleaning rather than changing the answer to `True`.

In [ ]:
# Write expressions that check the actual values in clean.
validation_checks = {
    "unique_ids": ____,
    "positive_measurements": ____,
    "allowed_categories": ____
}
display(validation_checks)

### Can someone reopen the saved table?
**Learn:** To save and reopen a separate table:
```python
parcels.to_csv("parcels.csv", index=False)
opened = pd.read_csv("parcels.csv")
print(opened.shape == parcels.shape)
print(opened.columns.tolist() == parcels.columns.tolist())
pd.testing.assert_frame_equal(
    opened.reset_index(drop=True),
    parcels.reset_index(drop=True),
    check_dtype=False
)
```
`index=False` leaves out pandas' row labels. `reset_index(drop=True)` gives the two tables comparable row labels without adding a column. `assert_frame_equal` checks their values and raises an error if they disagree; no output means it passed. `check_dtype=False` allows equivalent values with different stored types after reloading a CSV.

**Code:** Export `scaled` as `week01-campus-bike-trips-clean.csv` without an index. Read it into `reloaded`. Display the first rows, compare dimensions and column names, and check measurement values against `scaled`.

**Check:** Find the CSV in Colab's Files panel. It should have the five original columns and three added columns. Keep it for your own use; submit only the notebook.

In [ ]:
# Save scaled as week01-campus-bike-trips-clean.csv without row labels.
scaled.to_csv(____, index=____)

In [ ]:
reloaded = pd.read_csv(____)
display(reloaded.head())
print("Same dimensions:", ____)
print("Same columns:", ____)
# Compare table values, allowing equivalent stored types after CSV reload.
pd.testing.assert_frame_equal(____, ____, check_dtype=False)

## Written responses: A, B, and C · 3 points

**Student answer area:** edit the three Markdown cells below. Each response is two sentences and is manually graded for 1 point. Double-click a cell to edit it, replace the answer placeholder, then press **Shift+Enter** to display your answer.

Submit these answers inside this notebook only. The instructor can review them together in Gradescope's first three results; you do not need to copy them anywhere else.

### Response A: Cleaning decisions and consequences

**Question (2 sentences):** Report before/after row counts and which rules removed rows. How could excluding incomplete trips affect your supervisor's conclusion?

**Your answer:**

_Write your two sentences here._

### Response B: Normalization and standardization

**Question (2 sentences):** What does a min-max value of 0.5 mean? What does a z-score of 0 mean, and why are these different reference points?

**Your answer:**

_Write your two sentences here._

### Response C: Validation and remaining limitations

**Question (2 sentences):** Describe one observed validation or reload comparison and what it establishes. What is one problem that it cannot rule out?

**Your answer:**

_Write your two sentences here._

## Save, rerun, and submit
1. Save your Drive copy. Choose **Runtime → Restart session**, then **Runtime → Run all**. The loading cell reads the hosted CSV again; no upload is needed. Fix errors and finish Responses A, B, and C in the marked **Written responses** area.
2. Save again, then choose **File → Download → Download .ipynb**. Find the file on your computer, usually in Downloads.
3. Upload the notebook to **Gradescope → Lab02** and submit. Include your code, outputs, and written responses in this one file. Do not submit a CSV or copy your responses into separate questions.

The lab is 20 points: 17 automatic code points and 3 points for instructor review of Responses A, B, and C.